# 08_suspect_visualize

원본 Colab 셀에서 분리. (`#@title 의심 항목 시각화 (GT 초록 vs 예측 빨강)`)


In [ ]:
#@title [매 세션 1] rfdetr 설치
#@markdown 런타임이 끊기면 설치된 패키지가 전부 사라지므로, 매 세션 rfdetr을 다시 설치해야 함
!pip install -q "rfdetr[train,loggers]"                  # RF-DETR 학습(train)+로깅(loggers) 의존성

In [ ]:
#@title [매 세션 2] 드라이브 마운트 + 경로 자동 탐색
#@markdown 세션마다 드라이브 연결이 끊기므로 재마운트 필요. 데이터·zip이 전부 드라이브에 있어 경로부터 다시 잡아야 함
from google.colab import drive                           # 코랩↔드라이브 연결 도구
drive.mount('/content/drive')                             # 드라이브 마운트

import os, glob                                          # 경로·탐색 도구
CANDS = [                                                # 흔한 후보 경로 먼저
    '/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/sprint_ai_project1_data',
    '/content/drive/MyDrive/ai12-level1-project/sprint_ai_project1_data',
]
DATA_ROOT = next((c for c in CANDS if os.path.exists(c)), None)   # 존재하는 첫 경로 채택
if DATA_ROOT is None:                                    # 후보에 없으면 전체 재귀 검색
    hits = glob.glob('/content/drive/MyDrive/**/sprint_ai_project1_data', recursive=True)
    DATA_ROOT = hits[0] if hits else None
assert DATA_ROOT, "sprint_ai_project1_data를 못 찾음"     # 못 찾으면 중단
PROJ_ROOT = os.path.dirname(DATA_ROOT)                   # zip·백업 공통 상위 경로

TEST_IMG = os.path.join(DATA_ROOT, 'test_images')        # 제출용 842장(추론 때 사용)
print("DATA_ROOT:", DATA_ROOT)                           # 채택 경로 확인
print("PROJ_ROOT:", PROJ_ROOT)

In [ ]:
#@title [매 세션 3] dataset_74 zip 복원
import os                                                     # 경로 확인 도구

ZIP = os.path.join(PROJ_ROOT, "dataset_74_5fold.zip")         # 74클래스 5-fold zip(드라이브)
print("zip 존재:", os.path.exists(ZIP))                        # True 확인

!cp "$ZIP" /content/                                           # 드라이브 → 로컬 복사(속도 위해)
!unzip -qo /content/dataset_74_5fold.zip -d /content/dataset_74  # 압축 해제(-o=덮어쓰기)
print("복원 fold:", sorted(d for d in os.listdir("/content/dataset_74") if d.startswith("fold")))

In [ ]:
#@title 의심 항목 시각화 (GT 초록 vs 예측 빨강)
#@markdown valid_errors_74cls.csv에서 "분류오류" 7건 + "박스오류(동일클래스)" 15건을 골라, 해당 이미지에 GT 박스(초록)와 모델 예측 박스(빨강)를 겹쳐 그림. 라벨 오류인지 모델 오류인지 육안 판단용

# ============================================================
# [1] 한글 폰트 + import
# ============================================================
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1        # 나눔고딕 설치 (제목 한글 표시용)

import os, re, gc, json, glob                                # 경로·정규식·메모리·json·검색 도구
import unicodedata                                           # NFC/NFD 정규화 도구
import numpy as np                                            # 배열 도구
import pandas as pd                                           # CSV 읽기 도구
import torch                                                  # GPU 정리 도구
import matplotlib.pyplot as plt                               # 이미지 출력 도구
import matplotlib.font_manager as fm                          # 폰트 등록 도구
from PIL import Image, ImageDraw                              # 이미지 열기·박스 그리기 도구
from pathlib import Path                                      # 경로 객체 도구
from rfdetr import RFDETRMedium                                # 모델 클래스

for p in glob.glob("/usr/share/fonts/truetype/nanum/NanumGothic*.ttf"):
    fm.fontManager.addfont(p)                                 # matplotlib에 폰트 등록
plt.rc("font", family="NanumGothic")                          # 한글 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

# ============================================================
# [2] 경로 + 설정
# ============================================================
TEAM_ROOT = Path("/content/drive/MyDrive/1팀 공유 문서")
USER_ROOT = TEAM_ROOT / "김태윤"
DATASET = Path("/content/dataset_74")                         # 복원된 5-fold (앞 셀에서 이미 복원됨)
ERR_CSV = USER_ROOT / "error_analysis_74cls" / "valid_errors_74cls.csv"
LABEL_MAP_PATH = USER_ROOT / "label_map_74.json"

def find_dir(parent, keyword):
    key = unicodedata.normalize("NFC", keyword)               # NFC 정규화 후 부분일치 탐색
    for name in os.listdir(parent):
        if key in unicodedata.normalize("NFC", name) and (parent / name).is_dir():
            return parent / name
    return None

MODEL_DIR = find_dir(find_dir(USER_ROOT, "outputs_74"), "베스트모델")
assert MODEL_DIR is not None

NUM_CLASSES, RESOLUTION = 74, 576
PRED_THRESHOLD = 0.30                                          # 오류분석 때와 동일 threshold

# 시각화 대상 오류 유형 (라벨 오류 의심군)
TARGET_TYPES = ["분류오류", "박스오류(동일클래스)", "박스오류(클래스도틀림)"]

# ============================================================
# [3] 오류 목록 로드
# ============================================================
err = pd.read_csv(ERR_CSV)
targets = err[err["error_type"].isin(TARGET_TYPES)].copy()     # 의심 항목만 필터
targets = targets.sort_values(["error_type", "iou"])           # IoU 낮은 순 = 의심 강한 순
print(f"시각화 대상: {len(targets)}건")
print(targets["error_type"].value_counts().to_string())

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    id2kcode = {int(k): int(v) for k, v in json.load(f).items()}

# ============================================================
# [4] fold별로 묶어서 처리 (모델 로드 5번만)
# ============================================================
def draw_all_boxes(img_path, gt_boxes, pred_boxes):
    # 한 이미지에 GT(초록)와 예측(빨강) 박스를 전부 그리는 도구 #gt_boxes=[(xyxy,kcode)], pred_boxes=[(xyxy,kcode,score)]
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    for bbox, kcode in gt_boxes:                               # GT 전부 초록
        draw.rectangle(bbox, outline="lime", width=6)
        draw.text((bbox[0] + 5, bbox[1] + 5), f"GT:{kcode}", fill="lime")

    for bbox, kcode, score in pred_boxes:                      # 예측 전부 빨강
        draw.rectangle(bbox, outline="red", width=4)
        draw.text((bbox[0] + 5, bbox[3] - 20), f"P:{kcode} {score:.2f}", fill="red")

    return img

results = []                                                    # (이미지, 제목) 누적

for fi in sorted(targets["fold"].unique()):                     # fold 단위로 순회
    fold_targets = targets[targets["fold"] == fi]
    valid_dir = DATASET / f"fold{int(fi)}" / "valid"

    with open(valid_dir / "_annotations.coco.json", "r", encoding="utf-8") as f:
        coco = json.load(f)
    id2file = {im["id"]: im["file_name"] for im in coco["images"]}
    file2id = {v: k for k, v in id2file.items()}                # 파일명 -> image_id 역방향

    gt_by_image = {}                                             # 이미지별 GT 전부
    for ann in coco["annotations"]:
        x, y, w, h = ann["bbox"]
        gt_by_image.setdefault(ann["image_id"], []).append(
            ([x, y, x + w, y + h], id2kcode[ann["category_id"]])
        )

    ckpt = next(MODEL_DIR.glob(f"checkpoint_best_total_{int(fi):02d}_74.pth"))
    print(f"\n[fold {int(fi)}] 모델 로드: {ckpt.name} / 대상 {fold_targets['file_name'].nunique()}장")
    model = RFDETRMedium(pretrain_weights=str(ckpt), num_classes=NUM_CLASSES, resolution=RESOLUTION)

    for fname in fold_targets["file_name"].unique():             # 대상 이미지만 추론
        fpath = valid_dir / fname
        img = Image.open(fpath).convert("RGB")
        det = model.predict(img, threshold=PRED_THRESHOLD)

        pred_boxes = []
        for i in range(len(det)):
            pred_boxes.append((det.xyxy[i].tolist(), id2kcode[int(det.class_id[i])], float(det.confidence[i])))

        gt_boxes = gt_by_image.get(file2id[fname], [])
        vis = draw_all_boxes(fpath, gt_boxes, pred_boxes)

        # 이 이미지에 걸린 오류 요약을 제목으로
        rows = fold_targets[fold_targets["file_name"] == fname]
        desc = " | ".join(
            f"{r.error_type} GT:{int(r.gt_kcode)}→P:{int(r.pred_kcode) if pd.notna(r.pred_kcode) else '-'} IoU:{r.iou}"
            for r in rows.itertuples()
        )
        results.append((vis, f"fold{int(fi)} | {fname}\n{desc}"))

    del model; gc.collect(); torch.cuda.empty_cache()            # 다음 fold 대비 정리

# ============================================================
# [5] 출력 (1장씩 크게)
# ============================================================
for vis, title in results:
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(vis)
    ax.set_title(title, fontsize=8)
    ax.axis("off")
    plt.tight_layout()
    plt.show()